In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "ALQ_D.xpt"
csv_file = DATA_DIR / "ALQ_D_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Keep untouched source for fidelity comparison
source_df = df.copy()


print("=" * 80)
print("NHANES 2005-2006 ALQ_D XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. STRUCTURE CHECK
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "ALQ101",
    "ALQ110",
    "ALQ120Q",
    "ALQ120U",
    "ALQ130",
    "ALQ140Q",
    "ALQ140U",
    "ALQ150"
]


if len(df) != 4773:
    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 9:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "ALQ_D columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: ALQ_D structure = "
    "4,773 rows x 9 columns."
)


# ============================================================
# 4. KNOWN PANDAS/XPORT ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{int(true_zero_counts.sum()):,}"
)

print(
    "Tiny artifact occurrences:",
    f"{int(tiny_counts.sum()):,}"
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 5. CDC-VALIDATED ZERO VARIABLES
# ============================================================

#
# ALQ120Q:
# CDC instruction = ENTER '0' FOR NEVER
#
# ALQ140Q:
# CDC instruction = ENTER '0' FOR NONE
#

EXPECTED_TINY_COUNTS = {

    "ALQ120Q": 925,

    "ALQ140Q": 1775
}


ZERO_ALLOWED_COLUMNS = list(
    EXPECTED_TINY_COUNTS.keys()
)


# ============================================================
# 6. SAFETY CHECK
# ============================================================

actual_tiny_columns = set(
    tiny_counts[
        tiny_counts > 0
    ].index
)


approved_tiny_columns = set(
    ZERO_ALLOWED_COLUMNS
)


unexpected = (
    actual_tiny_columns
    -
    approved_tiny_columns
)


if unexpected:

    raise ValueError(
        "STOP: Tiny values found in unexpected "
        "ALQ_D variables:\n"
        + str(sorted(unexpected))
    )


missing_expected = (
    approved_tiny_columns
    -
    actual_tiny_columns
)


if missing_expected:

    raise ValueError(
        "STOP: Expected tiny-value variables missing:\n"
        + str(sorted(missing_expected))
    )


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        tiny_counts[col]
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


if int(tiny_counts.sum()) != 2700:

    raise ValueError(
        "Unexpected total tiny-value count."
    )


print(
    "\nPASS: All 2,700 tiny values occur only "
    "in CDC zero-permitted ALQ fields."
)


# ============================================================
# 7. RESTORE VALIDATED ZEROS
# ============================================================

print("\n--- ZERO RESTORATION ---")


for col in ZERO_ALLOWED_COLUMNS:

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    count = int(
        mask.sum()
    )


    df.loc[
        mask,
        col
    ] = 0.0


    print(
        f"{col:<10}: "
        f"{count:,} tiny -> 0"
    )


# ============================================================
# 8. VERIFY RESTORED ZEROS
# ============================================================

print("\n--- RESTORED ZERO COUNTS ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_zero = int(
        df[col]
        .eq(0)
        .sum()
    )


    print(
        f"{col:<10}: {actual_zero:,}"
    )


    if actual_zero != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,} zeros, "
            f"found {actual_zero:,}."
        )


print(
    "PASS: All 2,700 validated zeros restored."
)


# ============================================================
# 9. FINAL TINY CHECK
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nRemaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny-value artifacts remain."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 10. CONFIRM NO GENUINE DECIMAL VARIABLES
# ============================================================

print("\n--- FRACTIONAL VALUE CHECK ---")


fraction_failures = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
    )


    if values.empty:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional.sum()
    )


    print(
        f"{col:<10}: "
        f"fractional values = {count:,}"
    )


    if count != 0:

        fraction_failures.append(
            col
        )


if fraction_failures:

    raise ValueError(
        "STOP: Genuine decimal values found in:\n"
        + str(fraction_failures)
    )


print(
    "PASS: ALQ_D contains no genuine decimal variables."
)


# ============================================================
# 11. SEQN VALIDATION
# ============================================================

print("\n--- SEQN ---")


print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(df["SEQN"].isna().sum()) != 0
    or
    int(df["SEQN"].duplicated().sum()) != 0
    or
    df["SEQN"].min() != 31130
    or
    df["SEQN"].max() != 41473
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 12. CDC CHECK - ALQ101
# ============================================================

EXPECTED_ALQ101 = {
    1: 2994,
    2: 1355,
    9: 5
}


actual = (
    df["ALQ101"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if not (
    actual == EXPECTED_ALQ101
    and
    int(df["ALQ101"].isna().sum()) == 419
):

    raise ValueError(
        "ALQ101 differs from CDC.\n"
        f"Actual: {actual}"
    )


print(
    "\nPASS: ALQ101 matches CDC."
)


# ============================================================
# 13. CDC CHECK - ALQ110
# ============================================================

EXPECTED_ALQ110 = {
    1: 753,
    2: 604,
    9: 3
}


actual = (
    df["ALQ110"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if not (
    actual == EXPECTED_ALQ110
    and
    int(df["ALQ110"].isna().sum()) == 3413
):

    raise ValueError(
        "ALQ110 differs from CDC.\n"
        f"Actual: {actual}"
    )


print(
    "PASS: ALQ110 matches CDC."
)


# ============================================================
# 14. CDC CHECK - ALQ120Q
# ============================================================

s = df["ALQ120Q"]


if not (
    int(
        s.between(
            0,
            365
        ).sum()
    )
    == 3740

    and

    int(
        s.eq(999)
        .sum()
    )
    == 7

    and

    int(
        s.eq(777)
        .sum()
    )
    == 0

    and

    int(
        s.eq(0)
        .sum()
    )
    == 925

    and

    int(
        s.isna()
        .sum()
    )
    == 1026
):

    raise ValueError(
        "ALQ120Q differs from CDC/source."
    )


print(
    "PASS: ALQ120Q matches CDC/source."
)

print(
    "      Zero / Never = 925"
)


# ============================================================
# 15. CDC CHECK - ALQ120U
# ============================================================

EXPECTED_ALQ120U = {
    1: 1182,
    2: 698,
    3: 935
}


actual = (
    df["ALQ120U"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if not (
    actual == EXPECTED_ALQ120U
    and
    int(df["ALQ120U"].isna().sum()) == 1958
):

    raise ValueError(
        "ALQ120U differs from CDC."
    )


print(
    "PASS: ALQ120U matches CDC."
)


# ============================================================
# 16. CDC CHECK - ALQ130
# ============================================================

s = df["ALQ130"]


if not (
    int(
        s.between(
            1,
            32
        ).sum()
    )
    == 2820

    and

    int(
        s.eq(999)
        .sum()
    )
    == 2

    and

    int(
        s.eq(777)
        .sum()
    )
    == 0

    and

    int(
        s.isna()
        .sum()
    )
    == 1951
):

    raise ValueError(
        "ALQ130 differs from CDC."
    )


print(
    "PASS: ALQ130 matches CDC."
)


# ============================================================
# 17. CDC CHECK - ALQ140Q
# ============================================================

s = df["ALQ140Q"]


if not (
    int(
        s.between(
            0,
            365
        ).sum()
    )
    == 2811

    and

    int(
        s.eq(9999)
        .sum()
    )
    == 11

    and

    int(
        s.eq(7777)
        .sum()
    )
    == 0

    and

    int(
        s.eq(0)
        .sum()
    )
    == 1775

    and

    int(
        s.isna()
        .sum()
    )
    == 1951
):

    raise ValueError(
        "ALQ140Q differs from CDC/source."
    )


print(
    "PASS: ALQ140Q matches CDC/source."
)

print(
    "      Zero / None = 1,775"
)


# ============================================================
# 18. CDC CHECK - ALQ140U
# ============================================================

EXPECTED_ALQ140U = {
    1: 260,
    2: 168,
    3: 608
}


actual = (
    df["ALQ140U"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if not (
    actual == EXPECTED_ALQ140U
    and
    int(df["ALQ140U"].isna().sum()) == 3737
):

    raise ValueError(
        "ALQ140U differs from CDC."
    )


print(
    "PASS: ALQ140U matches CDC."
)


# ============================================================
# 19. CDC CHECK - ALQ150
# ============================================================

EXPECTED_ALQ150 = {
    1: 618,
    2: 3128,
    9: 1
}


actual = (
    df["ALQ150"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if not (
    actual == EXPECTED_ALQ150
    and
    int(df["ALQ150"].isna().sum()) == 1026
):

    raise ValueError(
        "ALQ150 differs from CDC."
    )


print(
    "PASS: ALQ150 matches CDC."
)


# ============================================================
# 20. BUILD EXPECTED FIDELITY SOURCE
# ============================================================

#
# Expected source =
#
# original XPT
# +
# ONLY:
#
# ALQ120Q tiny -> 0
# ALQ140Q tiny -> 0
#

expected_df = source_df.copy()


for col in ZERO_ALLOWED_COLUMNS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 21. VALUE FIDELITY BEFORE FORMATTING
# ============================================================

print("\n--- VALUE FIDELITY BEFORE FORMATTING ---")


value_failures = []


for col in expected_df.columns:

    if not np.allclose(
        expected_df[col].to_numpy(),
        df[col].to_numpy(),
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        value_failures.append(
            col
        )


if value_failures:

    raise ValueError(
        "Unexpected value changes in:\n"
        + str(value_failures)
    )


print(
    "PASS: All values match original XPT "
    "after ONLY the two validated zero restorations."
)


# ============================================================
# 22. FORMAT ALL FIELDS AS WHOLE NUMBERS
# ============================================================

#
# We already proved all nonmissing values are
# exact whole numbers.
#
# Therefore:
#
# SEQN:
# 31130.0 -> 31130
#
# ALQ101:
# 1.0 -> 1
#
# ALQ120Q:
# 0.0 -> 0
#
# ALQ140Q:
# 9999.0 -> 9999
#
# Missing:
# NaN -> blank
#

for col in df.columns:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: SEQN/questionnaire fields formatted "
    "without unnecessary .0."
)


# ============================================================
# 23. EXPORT CSV
# ============================================================

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 24. RE-READ CSV FOR NUMERIC FIDELITY
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 25. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- XPT -> CSV NUMERIC FIDELITY ---")


numeric_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )

    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    if not np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        numeric_failures.append(
            col
        )


if numeric_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(numeric_failures)
    )


print(
    "PASS: All 9 CSV columns numerically "
    "match expected XPT values."
)


# ============================================================
# 26. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )

    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\nMissing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 27. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)
    raw_rows = list(reader)


# ============================================================
# 28. STRICT RAW FORMAT FIDELITY
# ============================================================

#
# Every field in ALQ_D is an exact whole-number
# identifier/code/count/unit field.
#
# Compare directly with:
#
# str(int(expected XPT value))
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(raw_rows):

    for col in expected_df.columns:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(raw_examples) < 20:

                raw_examples.append(
                    (
                        i + 2,
                        col,
                        expected_text,
                        actual_text
                    )
                )


print(
    "\nRaw-format mismatches:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "Examples:",
        raw_examples
    )

    raise ValueError(
        "ALQ_D raw formatting changed."
    )


print(
    "PASS: Raw CSV formatting exactly matches "
    "expected questionnaire representation."
)


# ============================================================
# 29. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(dot_zero_examples) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nValues ending '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        "Examples:",
        dot_zero_examples
    )

    raise ValueError(
        "Unnecessary .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 values."
)


# ============================================================
# 30. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = value.lower()


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(scientific_examples) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        "Examples:",
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 31. FINAL TINY TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


if "5.397605346934028e-79" in csv_text:

    raise ValueError(
        "Tiny XPORT artifact remains in CSV."
    )


print(
    "PASS: Tiny artifact absent from CSV."
)


# ============================================================
# 32. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("ALQ_D CONVERSION COMPLETE")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "Validated tiny -> zero:",
    "2,700"
)

print(
    "ALQ120Q zeros:",
    "925"
)

print(
    "ALQ140Q zeros:",
    "1,775"
)

print(
    "Genuine decimal variables:",
    0
)

print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)

print(
    "Missing-position differences:",
    "0 expected"
)

print(
    "Raw-format differences:",
    "0 expected"
)

print(
    "Values ending .0:",
    "0 expected"
)

print(
    "Scientific notation:",
    "0 expected"
)

NHANES 2005-2006 ALQ_D XPT -> CSV
Rows: 4,773
Columns: 9
PASS: ALQ_D structure = 4,773 rows x 9 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 2,700

Tiny values by variable:
ALQ140Q    1775
ALQ120Q     925

PASS: All 2,700 tiny values occur only in CDC zero-permitted ALQ fields.

--- ZERO RESTORATION ---
ALQ120Q   : 925 tiny -> 0
ALQ140Q   : 1,775 tiny -> 0

--- RESTORED ZERO COUNTS ---
ALQ120Q   : 925
ALQ140Q   : 1,775
PASS: All 2,700 validated zeros restored.

Remaining tiny artifacts: 0
PASS: No tiny-value artifacts remain.

--- FRACTIONAL VALUE CHECK ---
SEQN      : fractional values = 0
ALQ101    : fractional values = 0
ALQ110    : fractional values = 0
ALQ120Q   : fractional values = 0
ALQ120U   : fractional values = 0
ALQ130    : fractional values = 0
ALQ140Q   : fractional values = 0
ALQ140U   : fractional values = 0
ALQ150    : fractional values = 0
PASS: ALQ_D contains no genuine decimal variables.

--- SEQN ---
Missing: 0
Duplicat